# 02. Multi-Agent Handoffs

This notebook shows specialization and delegation. One agent cleans OCR, another extracts metadata, and a third writes a short summary.

## Concepts
- Agents as tools
- Handoffs
- Pipeline design

## Dataset
A noisy OCR fragment is included in `notebooks/common.py`.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from common import LETTER_TEXTS
from agents import Agent, Runner

raw_text = LETTER_TEXTS[2]['text']
print(raw_text)

## Step 1: Define the specialist agents

The cleaner focuses on transcription quality. The extractor focuses on structured facts. The summarizer focuses on prose.

In [ ]:
cleaner = Agent(
    name='OCR Cleaner',
    instructions=(
        'Fix obvious OCR mistakes while preserving original meaning. '
        'Do not invent new information.'
    ),
)

extractor = Agent(
    name='Metadata Extractor',
    instructions='Extract people, places, dates, and key facts from cleaned historical text.',
)

summarizer = Agent(
    name='Historical Summarizer',
    instructions='Write a two-sentence summary grounded in the provided cleaned text and extracted facts.',
)

cleaner, extractor, summarizer

## Step 2: Use agents as tools

The coordinator can call other agents with `as_tool`. That is often the easiest way to teach agents-as-tools.

In [ ]:
cleaner_tool = cleaner.as_tool('clean_ocr', 'Clean OCR text while preserving meaning.')
extractor_tool = extractor.as_tool('extract_metadata', 'Extract structured metadata from cleaned text.')
summarizer_tool = summarizer.as_tool('summarize_history', 'Write a short summary based on cleaned text and facts.')

print(cleaner_tool.name, extractor_tool.name, summarizer_tool.name)

## Step 3: Coordinator agent

The coordinator orchestrates the work. This gives students a concrete view of delegation without hiding the steps.

In [ ]:
coordinator = Agent(
    name='Pipeline Coordinator',
    instructions=(
        'First clean OCR, then extract metadata, then write a short summary. '
        'Always prefer evidence over guessing.'
    ),
    tools=[cleaner_tool, extractor_tool, summarizer_tool],
)

coordinator

## Step 4: Run the pipeline

This call is fully runnable once the API key is set. Students can inspect the result and compare the intermediate outputs conceptually.

In [ ]:
result = Runner.run_sync(coordinator, raw_text)
print(result.final_output)


## Exercise

Add a fourth agent that classifies whether the text looks like a letter, notice, or newspaper excerpt.

## Solution

Create `classifier = Agent(...)`, wrap it with `classifier.as_tool(...)`, add it to `coordinator.tools`, and update the instructions to call it before summarization.